In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
import torch

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from src.trainer import EWCTrainer, SimpleTrainer
from src.data_utils import get_mnist_tasks, _extract_targets, get_context_sets
from src.utils.general import InContextHead, set_seed
from src import models

### Task Incremental Learning

In [4]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

context_sets = get_context_sets(test_tasks)
head = InContextHead(context_sets, 10, device="cuda")
head.set_context(0)
model = models.get_mnist_model(head=head, device="cuda")

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[3, 4], [8, 9], [6, 7], [0, 5], [1, 2]]


In [5]:
trainer = SimpleTrainer(model)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
trainer.test(test_tasks[0:1])

Training Epochs: 100%|██████████████████| 3/3 [00:03<00:00,  1.07s/it, train_loss=0.0107, val_loss=0.0425, val_acc=0.988]


Test Results: [(0.0209, 0.993)]


[(0.0209, 0.993)]

In [7]:
ewc_trainer = EWCTrainer(
    trainer.model,
    paradigm="TIL",
)

ewc_trainer.compute_fisher_information(trainer.model, train_tasks[0], loss_fn=torch.nn.CrossEntropyLoss())

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    ewc_trainer.train(train, val, batch_size=256, context_id=i)
    ewc_trainer.test(test_tasks[0 : i + 1], context_list=list(range(i + 1)))

Training Epochs: 100%|████████████████████████████████████| 5/5 [00:06<00:00,  1.31s/it, val_loss=0.0964, val_acc=0.9740]


Test Results: [(0.0355, 0.994), (0.0626, 0.9788)]


Training Epochs: 100%|████████████████████████████████████| 5/5 [00:06<00:00,  1.34s/it, val_loss=0.0054, val_acc=0.9980]


Test Results: [(0.051, 0.993), (0.0744, 0.9803), (0.01, 0.9955)]


Training Epochs: 100%|████████████████████████████████████| 5/5 [00:06<00:00,  1.25s/it, val_loss=0.0652, val_acc=0.9795]


Test Results: [(0.061, 0.991), (0.0709, 0.9773), (0.0099, 0.9965), (0.0384, 0.9856)]


Training Epochs: 100%|████████████████████████████████████| 5/5 [00:06<00:00,  1.39s/it, val_loss=0.0382, val_acc=0.9898]


Test Results: [(0.0689, 0.9915), (0.0677, 0.9793), (0.0099, 0.996), (0.04, 0.9845), (0.0416, 0.9917)]


### Domain Incremental Learning

In [8]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

model = models.get_mnist_model(device="cuda")

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[3, 4], [8, 9], [6, 7], [0, 5], [1, 2]]


In [9]:
def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
    """Map the global label to the in context label."""
    return labels % 2

In [10]:
trainer = SimpleTrainer(model, paradigm="DIL", domain_map_fn=domain_map_fn)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
trainer.test(test_tasks[0:1])

Training Epochs: 100%|██████████████████| 3/3 [00:02<00:00,  1.09it/s, train_loss=0.0101, val_loss=0.0421, val_acc=0.987]


Test Results: [(0.0208, 0.994)]


[(0.0208, 0.994)]

In [13]:
ewc_trainer = EWCTrainer(
    trainer.model,
    paradigm="DIL",
    domain_map_fn=domain_map_fn
)

ewc_trainer.compute_fisher_information(trainer.model, train_tasks[0], loss_fn=torch.nn.CrossEntropyLoss())

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    ewc_trainer.train(train, val, batch_size=256)
    ewc_trainer.test(test_tasks[0 : i + 1])

Training Epochs: 100%|████████████████████████████████████| 5/5 [00:06<00:00,  1.30s/it, val_loss=0.1019, val_acc=0.9698]


Test Results: [(4.3186, 0.118), (0.0891, 0.9723)]


Training Epochs: 100%|████████████████████████████████████| 5/5 [00:06<00:00,  1.34s/it, val_loss=0.0048, val_acc=0.9980]


Test Results: [(2.7037, 0.5582), (2.1506, 0.6536), (0.0065, 0.9975)]


Training Epochs: 100%|████████████████████████████████████| 5/5 [00:06<00:00,  1.23s/it, val_loss=0.0648, val_acc=0.9750]


Test Results: [(2.1978, 0.5095), (2.3903, 0.5088), (0.7717, 0.7578), (0.0394, 0.9861)]


Training Epochs: 100%|████████████████████████████████████| 5/5 [00:06<00:00,  1.38s/it, val_loss=0.0433, val_acc=0.9863]


Test Results: [(3.2045, 0.5055), (3.1099, 0.4534), (1.8977, 0.6133), (1.756, 0.7366), (0.0495, 0.988)]


### Class Incremental Learning

In [25]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=9810)

model = models.get_mnist_model(device="cuda", seed=9810)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[2, 9], [1, 8], [4, 5], [0, 3], [6, 7]]


In [26]:
trainer = SimpleTrainer(model, seed=9810)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
trainer.test(test_tasks[0:1])

Training Epochs: 100%|██████████████████| 3/3 [00:02<00:00,  1.15it/s, train_loss=0.0334, val_loss=0.0668, val_acc=0.974]


Test Results: [(0.0613, 0.9833)]


[(0.0613, 0.9833)]

In [ ]:
ewc_trainer = EWCTrainer(
    trainer.model,
    paradigm="CIL",
    lmbd=1e6,
)

ewc_trainer.compute_fisher_information(trainer.model, train_tasks[0], loss_fn=torch.nn.CrossEntropyLoss())

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    ewc_trainer.train(train, val, batch_size=256)
    ewc_trainer.test(test_tasks[0 : i + 1])

Training Epochs: 100%|████████████████████████████████████| 5/5 [00:07<00:00,  1.47s/it, val_loss=0.1778, val_acc=0.9570]


Test Results: [(4.4436, 0.0), (0.2003, 0.9734)]


Training Epochs: 100%|████████████████████████████████████| 5/5 [00:06<00:00,  1.33s/it, val_loss=1.0190, val_acc=0.5691]


Test Results: [(4.2659, 0.0), (1.5294, 0.1266), (0.7384, 0.7753)]


Training Epochs: 100%|███████████████████████████████████| 5/5 [00:07<00:00,  1.45s/it, val_loss=15.5376, val_acc=0.0000]


Test Results: [(2.6327, 0.0), (1.4127, 0.303), (0.9077, 0.8602), (14.7616, 0.0)]


Training Epochs: 100%|████████████████████████████████████| 5/5 [00:07<00:00,  1.52s/it, val_loss=8.4593, val_acc=0.0000]


Test Results: [(2.6442, 0.0), (1.4178, 0.3025), (0.9091, 0.8602), (14.7633, 0.0), (7.2405, 0.0)]
